# **[_CREATE TABLE AS and COPY INTO_](url)**

## Exploring the data source files

/Volumes/pysaprk_demo/default/users_parque

#### List of files in the give volume

In [0]:
spark.createDataFrame(dbutils.fs.ls('/Volumes/pysaprk_demo/default/users_parque')).show()

In [0]:
spark.read.format('parquet').load('/Volumes/pysaprk_demo/default/users_parque/users_01.parquet').show(2)

In [0]:
spark.read.format('parquet').load('/Volumes/pysaprk_demo/default/users_parque/').show(2)

#### Checking current catalog and schema

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema

#### Change the catalog and schema

In [0]:
%sql
-- select catalog
USE CATALOG pysaprk_demo;
-- select schema
USE SCHEMA demo;  
-- display the current catalog and schema
SELECT current_catalog() as catalog, current_schema() as schema

### **[_Batch Data Ingestion with CTAS and read_files()_](url)**

The CREATE TABLE AS (CTAS) statement is used to create and populate table using the results of query. This allows you to create a table and load it with data in a single step, streamlining data ingestion workflows.

### **Automatic Schema Inference for Parquet Files**

Apache Parquet is a columnar storage format optimized for analytical queries. It includes embedded schema metadata (e.g., column names and data types), which enables automatic schema inference when creating table from Parquet file. This eliminates the need for manual schema definations and simplifies the process of converting Parquet files into Delta format by leveraging the built-in schema metadata.

### CTAS with the read_files() Function

In [0]:
%sql
SELECT *
FROM read_files(
  "/Volumes/pysaprk_demo/default/users_parque/",
  format => "parquet"
  )
limit 5
;

In [0]:
%sql
-- Drop table if it exists for reproducibility
DROP TABLE IF EXISTS historical_users_bronze_ctas_rf;

-- Create the Delta table
CREATE TABLE historical_users_bronze_ctas_rf AS
SELECT *
FROM read_files(
    "/Volumes/pysaprk_demo/default/users_parque/",
  format => "parquet"
);

-- Preview the Delta table
SELECT * FROM historical_users_bronze_ctas_rf limit 2; 

In [0]:
%sql
-- Run the DESCRIBE TABLE EXTEND statement to view the column names, data types, and additional table metadata
DESCRIBE TABLE EXTENDED historical_users_bronze_ctas_rf; 

### **_[Managed vs External Tables in Databricks](url)_**

#### **_Managed Tables_**
1. Databricks manages both the data and metadata
2. Data is stored within databrick's managed storage
3. Dropping the table also deletes the data
4. Recommended for creating new table

#### **_External Tables_**
1. Databricks only manages the table metadata
2. Dropping the table does not delete the data
3. Support multiple formats, including Delta Lake
4. Idela for sharing data accross platforms or using external data 



In [0]:
# The code uses Python to ingest the parquet files

# 1. Read the parquet files into a Spark DataFrame
df = spark.read.parquet("/Volumes/pysaprk_demo/default/users_parque/")

# 2. Write to the Dataframe to a Delta table (overwrite the table if it exists)
df.write.mode("overwrite").saveAsTable("historical_users_bronze_python")

# 3. Read and view the table
users_bronze_table = spark.table("historical_users_bronze_python")
users_bronze_table.show(2)

### **[_Incremental Data Ingestion with COPY INTO_](url)**

COPY INTO is a databricks SQL command that allows you to load data from a file location into a Delta table. This operation is re-triable and idempotent, i.e., files in the source location that have already be loaded are skipped. This command is useful for when you need to load data into an existing Delta table.

In [0]:
%sql

--  -------------------------------------------------------
--  This cell returns an error
--  -------------------------------------------------------

--  DROP the table if it exists for reproducibility
DROP TABLE IF EXISTS historical_users_bronze_ci;

--  Create an empty table with the specified table schema (user_id | user_first_touch_point | email| name | age | gender | registration_date) with few missing columns
CREATE TABLE historical_users_bronze_ci (
  user_id STRING,
  email STRING,
  name STRING,
  age INT,
  gender STRING
);

-- Preview the table 
SELECT * FROM historical_users_bronze_ci;

-- Use the COPY INTO to populate Delta table
COPY INTO historical_users_bronze_ci
FROM "/Volumes/pysaprk_demo/default/users_parque/"
FILEFORMAT = PARQUET
;

In [0]:
%sql
-- Enable timestampNtz feature for the table
ALTER TABLE historical_users_bronze_ci 
SET TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported');

-- COPY INTO method using mergeSchema = 'true' option
COPY INTO historical_users_bronze_ci
FROM "/Volumes/pysaprk_demo/default/users_parque/"
FILEFORMAT = parquet
COPY_OPTIONS('mergeSchema' = 'true');

In [0]:
%sql
SELECT * FROM historical_users_bronze_ci limit 2

### Preemptively Handling Schema Evolution

Another way to ingest the schema files into a Delta table is to start by creating an empty table names **historical_users_bronze_ci_no_schema** then, add the COPY_OPTIONS ('mergeSchema' = 'true') option to enable schema evolution for the table.

Run the cell and confirm thet 10,000 rows were added to the Delta table

In [0]:
%sql
-- Drop the table if it exists for reproducibility
DROP TABLE IF EXISTS historical_users_bronze_ci_no_schema;

-- Create an empty table without the specified schema
CREATE TABLE historical_users_bronze_ci_no_schema;

-- Enable timestampNtz feature for the table
ALTER TABLE historical_users_bronze_ci_no_schema 
SET TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported');

-- USE COPY INTO to populate the Delta table
COPY INTO historical_users_bronze_ci_no_schema
FROM "/Volumes/pysaprk_demo/default/users_parque/"
FILEFORMAT = parquet
COPY_OPTIONS('mergeSchema' = 'true')
;

### **Idempotency (Incremental Ingestion)**
COPY INTO tracks the files it has previously ingested. If the command is run again, no additional data is ingested the files in the source directory haven't changed.

In [0]:
%sql
-- Let's run the COPY INTO command again and check if any data is  
COPY INTO historical_users_bronze_ci_no_schema
FROM "/Volumes/pysaprk_demo/default/users_parque/"
FILEFORMAT = parquet
COPY_OPTIONS('mergeSchema' = 'true')
;

In [0]:
%sql
-- Run the COPY INTO command after adding new files
COPY INTO historical_users_bronze_ci_no_schema
FROM "/Volumes/pysaprk_demo/default/users_parque/"
FILEFORMAT = parquet
COPY_OPTIONS('mergeSchema' = 'true')
;


### **[_Test Incremental Process with same data accross all the files_](url)**

In [0]:
%sql
-- Drop table if exists for reproducibility
DROP TABLE IF EXISTS historical_users_bronze_ci_incremental;

-- Create an empty table without the specified schema
CREATE TABLE historical_users_bronze_ci_incremental;

-- Enable timestampNtz feature for the table
ALTER TABLE historical_users_bronze_ci_incremental 
SET TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported');

-- Insert a data into the table using COPY INTO
COPY INTO historical_users_bronze_ci_incremental
FROM "/Volumes/pysaprk_demo/default/incremental_data_test/test_01.parquet"
FILEFORMAT = parquet
COPY_OPTIONS('mergeSchema' = 'true')
;

In [0]:
%sql
select * from historical_users_bronze_ci_incremental;

In [0]:
%sql
-- Inserting a new file into the table using COPY INTO
COPY INTO historical_users_bronze_ci_incremental
FROM "/Volumes/pysaprk_demo/default/incremental_data_test/"
FILEFORMAT = parquet
COPY_OPTIONS('mergeSchema' = 'true')
;
    
select * from historical_users_bronze_ci_incremental;

In [0]:
%sql
-- MERGE provides record-level deduplication using business keys
-- Unlike COPY INTO (file-level), MERGE handles duplicate records

-- Create a target table for MERGE example
DROP TABLE IF EXISTS historical_users_bronze_merge;
CREATE TABLE historical_users_bronze_merge AS
SELECT * FROM historical_users_bronze_ci_incremental WHERE 1=0;

-- First load: Insert initial data
MERGE INTO historical_users_bronze_merge AS target
USING (
  SELECT * FROM read_files(
    '/Volumes/pysaprk_demo/default/incremental_data_test/test_01.parquet',
    format => 'parquet'
  )
) AS source
ON target.user_id = source.user_id
WHEN MATCHED THEN
  UPDATE SET
    target.user_first_touch_point = source.user_first_touch_point,
    target.email = source.email,
    target.name = source.name,
    target.age = source.age,
    target.gender = source.gender,
    target.registration_date = source.registration_date
WHEN NOT MATCHED THEN
  INSERT (user_id, user_first_touch_point, email, name, age, gender, registration_date)
  VALUES (source.user_id, source.user_first_touch_point, source.email, source.name, source.age, source.gender, source.registration_date);

SELECT 'After first load:' as status, COUNT(*) as row_count FROM historical_users_bronze_merge;

In [0]:
%sql
-- Load all files from directory with MERGE
-- Notice: Even though 7 files contain the same record, only 1 row exists

MERGE INTO historical_users_bronze_merge AS target
USING (
  SELECT * FROM read_files(
    '/Volumes/pysaprk_demo/default/incremental_data_test/',
    format => 'parquet'
  )
) AS source
ON target.user_id = source.user_id
WHEN MATCHED THEN
  UPDATE SET
    target.user_first_touch_point = source.user_first_touch_point,
    target.email = source.email,
    target.name = source.name,
    target.age = source.age,
    target.gender = source.gender,
    target.registration_date = source.registration_date
WHEN NOT MATCHED THEN
  INSERT *;

SELECT 'After loading all files:' as status, COUNT(*) as row_count FROM historical_users_bronze_merge;

-- Show the actual data
SELECT * FROM historical_users_bronze_merge;

### **COPY INTO vs MERGE: Understanding Duplicate Handling**

#### **COPY INTO (File-Level Idempotency)**
* ✅ **Prevents**: Loading the same **file** multiple times
* ❌ **Does NOT prevent**: Duplicate **records** across different files
* **Use case**: Incremental file ingestion where source files are unique
* **Result with test data**: 7 files → 7 identical rows (1 per file)

#### **MERGE (Record-Level Deduplication)**
* ✅ **Prevents**: Duplicate **records** based on business key (e.g., `user_id`)
* ✅ **Handles**: Updates to existing records (upsert pattern)
* **Use case**: CDC, SCD Type 1, deduplication based on unique identifiers
* **Result with test data**: 7 files → 1 unique row (deduplicated by `user_id`)

#### **Comparison Table**

| Feature | COPY INTO | MERGE |
|---------|-----------|-------|
| Idempotency Level | File | Record |
| Tracks | File paths | Business keys |
| Duplicate Files | ✅ Skipped | ⚠️ Not tracked |
| Duplicate Records | ❌ All inserted | ✅ Deduplicated |
| Performance | Faster (bulk load) | Slower (row-level operations) |
| Updates | ❌ Not supported | ✅ WHEN MATCHED |
| Best For | Append-only logs | CDC, SCD, deduplication |